In [31]:
!pip install -q sentence-transformers lightgbm

In [32]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
from pathlib import Path
from sentence_transformers import SentenceTransformer


for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

train = pd.read_csv(Path("/kaggle/input/competitions/llm-classification-finetuning/train.csv"))
test = pd.read_csv(Path("/kaggle/input/competitions/llm-classification-finetuning/test.csv"))

print(train.head())
print(train.columns)
print(len(train))

train["label"] = (
    train["winner_model_a"] * 0 +
    train["winner_model_b"] * 1 +
    train["winner_tie"] * 2
)

/kaggle/input/competitions/llm-classification-finetuning/sample_submission.csv
/kaggle/input/competitions/llm-classification-finetuning/train.csv
/kaggle/input/competitions/llm-classification-finetuning/test.csv
       id             model_a              model_b  \
0   30192  gpt-4-1106-preview           gpt-4-0613   
1   53567           koala-13b           gpt-4-0613   
2   65089  gpt-3.5-turbo-0613       mistral-medium   
3   96401    llama-2-13b-chat  mistral-7b-instruct   
4  198779           koala-13b   gpt-3.5-turbo-0314   

                                              prompt  \
0  ["Is it morally right to try to have a certain...   
1  ["What is the difference between marriage lice...   
2  ["explain function calling. how would you call...   
3  ["How can I create a test set for a very rare ...   
4  ["What is the best way to travel from Tel-Aviv...   

                                          response_a  \
0  ["The question of whether it is morally right ...   
1  ["A marriag

In [ ]:
# Cell - Embedding
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss

embedder = SentenceTransformer("all-MiniLM-L6-v2")  # 384-dim embeddings
X_a = embedder.encode(train["prompt"] + " [SEP] " + train["response_a"], batch_size=64, show_progress_bar=True)
X_b = embedder.encode(train["prompt"] + " [SEP] " + train["response_b"], batch_size=64, show_progress_bar=True)


# Creates a dictionary of words, transform text to a numerical feature vector
X = X_a - X_b  # shape = (n_samples, 384)
y = train["label"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

#X_test = embedder.encode(test["prompt"] + " [SEP] " + train["response_b"], batch_size=64, show_progress_bar=True)



'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
No sentence-transformers model found with name sentence-transformers/all-MiniLM-L6-v2. Creating a new one with mean pooling.


In [ ]:
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

params = {
    "objective": "multiclass",
    "num_class": 3,
    "metric": "multi_logloss",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "seed": 42,
    "verbosity": -1
}

model = lgb.train(
    params,
    lgb_train,
    num_boost_round=500,
    valid_sets=[lgb_train, lgb_val],
    early_stopping_rounds=50,
    verbose_eval=50
)

In [ ]:
val_preds = model.predict(X_val)
print("Validation log loss:", log_loss(y_val, val_preds))

In [ ]:
X_test_a = embedder.encode(test["prompt"] + " [SEP] " + test["response_a"], batch_size=64)
X_test_b = embedder.encode(test["prompt"] + " [SEP] " + test["response_b"], batch_size=64)
X_test = X_test_a - X_test_b

test_preds = model.predict(X_test)  # shape = (n_samples, 3)
# Contains probabilities for win_a, win_b, tie

#id,winner_model_a,winner_model_b,winner_tie
submission = pd.DataFrame(test_preds, columns=target_cols)
submission["id"] = test["id"]
submission = submission[["id"] + target_cols]
submission.to_csv("submission.csv", index=False)